In [11]:
# Clean and prepare data
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
from pathlib import Path


pd.set_option('display.max_columns', None)

In [12]:
# Define project directory paths for data access



PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[2]

BASE_DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR    = BASE_DATA_DIR / "raw"
BRONZE_DIR = BASE_DATA_DIR / "bronze"
SILVER_DIR = BASE_DATA_DIR / "silver"

RAW_DIR.mkdir(parents=True, exist_ok=True)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://www.data.gouv.fr/api/1/datasets/r/2e3e44de-e584-4aa2-8148-670daf5617e1"


path_data_raw     = RAW_DIR / "PR17_BVot_T2_FE.txt"
path_rhone_bronze = BRONZE_DIR / "2017-pres-t2-commune-rhone-69-bronze.csv"
path_rhone_silver = SILVER_DIR / "2017-pres-t2-commune-rhone-69-silver.csv"
print(f"Raw data path: {path_data_raw}")
print(f"Bronze path:   {path_rhone_bronze}")
print(f"Silver path:   {path_rhone_silver}")


Raw data path: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/raw/PR17_BVot_T2_FE.txt
Bronze path:   /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/bronze/2017-pres-t2-commune-rhone-69-bronze.csv
Silver path:   /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/silver/2017-pres-t2-commune-rhone-69-silver.csv


In [13]:
# Check if file exists and create/download if missing
# Iterate through items and process each one
if not os.path.exists(path_data_raw):
    print(f"📥 Downloading source to RAW...")
    resp = requests.get(DATA_URL, timeout=180)
    resp.raise_for_status()
    with open(path_data_raw, "wb") as f:
        f.write(resp.content)
    print(f"✅ Saved to RAW: {path_data_raw}")
else:
    print(f"✅ Raw file already exists in {RAW_DIR}")


✅ Raw file already exists in /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/raw


In [14]:
# Group dataframe and apply multiple aggregation functions
# Load and read data from data sources
print("\n🛠 Processing RAW -> BRONZE...")

COMMON_FIELDS = [
    "Code du département", "Libellé du département",
    "Code de la circonscription", "Libellé de la circonscription",
    "Code de la commune", "Libellé de la commune",
    "Code du b.vote", "Inscrits", "Abstentions", "% Abs/Ins",
    "Votants", "% Vot/Ins", "Blancs", "% Blancs/Ins", "% Blancs/Vot",
    "Nuls", "% Nuls/Ins", "% Nuls/Vot", "Exprimés", "% Exp/Ins", "% Exp/Vot",
]
CAND_FIELDS = ["N°Panneau", "Sexe", "Nom", "Prénom", "Voix", "% Voix/Ins", "% Voix/Exp"]
N_CANDIDATES = 2

all_columns = [
    "Code du département", "Libellé du département",
    "Code de la circonscription", "Libellé de la circonscription",
    "Code de la commune", "Libellé de la commune",
    "Code du b.vote", "Inscrits", "Abstentions", "% Abs/Ins",
    "Votants", "% Vot/Ins", "Blancs", "% Blancs/Ins", "% Blancs/Vot",
    "Nuls", "% Nuls/Ins", "% Nuls/Vot", "Exprimés", "% Exp/Ins", "% Exp/Vot",
    "N°Panneau_1", "Sexe_1", "Nom_1", "Prénom_1", "Voix_1", "% Voix/Ins_1", "% Voix/Exp_1",
    "N°Panneau_2", "Sexe_2", "Nom_2", "Prénom_2", "Voix_2", "% Voix/Ins_2", "% Voix/Exp_2",
]

df_all = pd.read_csv(
    path_data_raw,
    sep=";",
    dtype=str,
    encoding="cp1252",
    engine="python",
    header=None,
    names=all_columns
)

df_all.columns = df_all.columns.str.strip()
df_all["Code du département"] = df_all["Code du département"].astype(str).str.strip().str.zfill(2)
df_filtered = df_all[df_all["Code du département"] == "69"].copy()

numeric_cols = ['Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés', 'Voix_1', 'Voix_2']
for col in numeric_cols:
    df_filtered[col] = df_filtered[col].astype(str).str.replace(',', '.').astype(float)

df_bronze = df_filtered.groupby('Libellé de la commune').agg({
    'Code du département': 'first',
    'Libellé du département': 'first',
    'Code de la circonscription': 'first',
    'Libellé de la circonscription': 'first',
    'Code de la commune': 'first',   
    'Inscrits': 'sum',
    'Abstentions': 'sum',
    'Votants': 'sum',
    'Blancs': 'sum',
    'Nuls': 'sum',
    'Exprimés': 'sum',
    'Voix_1': 'sum',
    'Voix_2': 'sum',
    'N°Panneau_1': 'first',
    'Sexe_1': 'first',
    'Nom_1': 'first',
    'Prénom_1': 'first',
    'N°Panneau_2': 'first',
    'Sexe_2': 'first',
    'Nom_2': 'first',
    'Prénom_2': 'first',
}).reset_index()

int_cols = [
    'Inscrits',
    'Abstentions',
    'Votants',
    'Blancs',
    'Nuls',
    'Exprimés',
    'Voix_1',
    'Voix_2'
]

for col in int_cols:
    df_bronze[col] = pd.to_numeric(df_bronze[col], errors='coerce').round().astype('Int64')


df_bronze['% Abs/Ins'] = (df_bronze['Abstentions'] / df_bronze['Inscrits'] * 100).round(2)
df_bronze['% Vot/Ins'] = (df_bronze['Votants'] / df_bronze['Inscrits'] * 100).round(2)
df_bronze['% Blancs/Ins'] = (df_bronze['Blancs'] / df_bronze['Inscrits'] * 100).round(2)
df_bronze['% Blancs/Vot'] = (df_bronze['Blancs'] / df_bronze['Votants'] * 100).round(2)
df_bronze['% Nuls/Ins'] = (df_bronze['Nuls'] / df_bronze['Inscrits'] * 100).round(2)
df_bronze['% Nuls/Vot'] = (df_bronze['Nuls'] / df_bronze['Votants'] * 100).round(2)
df_bronze['% Exp/Ins'] = (df_bronze['Exprimés'] / df_bronze['Inscrits'] * 100).round(2)
df_bronze['% Exp/Vot'] = (df_bronze['Exprimés'] / df_bronze['Votants'] * 100).round(2)
df_bronze['% Voix/Ins_1'] = (df_bronze['Voix_1'] / df_bronze['Inscrits'] * 100).round(2)
df_bronze['% Voix/Exp_1'] = (df_bronze['Voix_1'] / df_bronze['Exprimés'] * 100).round(2)
df_bronze['% Voix/Ins_2'] = (df_bronze['Voix_2'] / df_bronze['Inscrits'] * 100).round(2)
df_bronze['% Voix/Exp_2'] = (df_bronze['Voix_2'] / df_bronze['Exprimés'] * 100).round(2)

df_bronze["extraction_source_url"] = DATA_URL
df_bronze["ingestion_timestamp"] = datetime.now().isoformat()
df_bronze["source_file_name"] = os.path.basename(path_data_raw)

df_bronze.to_csv(path_rhone_bronze, index=False, sep=";", encoding="utf-8")
print(f"✅ BRONZE dataset created: {path_rhone_bronze} ({len(df_bronze)} rows)")


🛠 Processing RAW -> BRONZE...
✅ BRONZE dataset created: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/bronze/2017-pres-t2-commune-rhone-69-bronze.csv (280 rows)


In [15]:
# Load and read data from data sources
df_all = pd.read_csv(
    path_data_raw,
    sep=";",
    dtype=str,
    encoding="cp1252",
    engine="python",
    skiprows=1
)

print(f"Columns: {df_all.columns.tolist()}")
print(f"\nFirst row:")
print(df_all.iloc[0])
print(f"\nShape: {df_all.shape}")

Columns: ['01', 'Ain', '04', '4ème circonscription', '001', "L'Abergement-Clémenciat", '0001', '598', '100', '16,72', '498', '83,28', '37', '6,19', '7,43', '8', '1,34', '1,61', '453', '75,75', '90,96', '1', 'M', 'MACRON', 'Emmanuel', '272', '45,48', '60,04', '2', 'F', 'LE PEN', 'Marine', '181', '30,27', '39,96']

First row:
01                                            01
Ain                                          Ain
04                                            05
4ème circonscription        5ème circonscription
001                                          002
L'Abergement-Clémenciat    L'Abergement-de-Varey
0001                                        0001
598                                          209
100                                           32
16,72                                      15,31
498                                          177
83,28                                      84,69
37                                            21
6,19                                 